# JOAI2026 v21: 非線形後処理（Power Transform + Session Scaling）

### コンセプト
新しいモデルを学習せず、**v19bの予測値を最適に変形する**ことでスコアを改善。

### 3段階の後処理パイプライン
1. **Power Transformation**: `sign(y) × |y|^p` で谷を強調（アンサンブルの角丸め補正）
2. **Session Scaling**: mouse_id × day_n 単位で振幅を最適スケーリング
3. **組み合わせ最適化**: Power → Session の連鎖効果を検証

### 学習なし、OOFでgrid searchのみ

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
## Cell 2: Imports
import numpy as np
import pandas as pd
import os, warnings
from pathlib import Path
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')
print('v21: Post-Processing Pipeline')

In [ ]:
## Cell 3: データ + 全OOF/test読み込み
data_dir = Path('/content/drive/MyDrive/joai2026/data')
oof_dir = Path('/content/drive/MyDrive/joai2026/oof')

train_df = pd.read_csv(data_dir / 'train.csv')
test_df = pd.read_csv(data_dir / 'test.csv')
print(f'Train: {train_df.shape}, Test: {test_df.shape}')

train_df['day_n'] = train_df['sample_id'].str.extract(r'day_(\d+)').astype(int)
test_df['day_n'] = test_df['sample_id'].str.extract(r'day_(\d+)').astype(int)

y_true = train_df['lever'].values

# --- v10b ---
oof_v10b, test_v10b = None, None
for d in [oof_dir, data_dir]:
    p = d / 'oof_stacked_v10b.npy'
    if p.exists() and oof_v10b is None:
        oof_v10b = np.load(p)
    p = d / 'test_stacked_v10b.npy'
    if p.exists() and test_v10b is None:
        test_v10b = np.load(p)
print(f'v10b: oof={oof_v10b.shape}, test={test_v10b.shape}')

# --- v17a MLP ---
oof_mlp17, test_mlp17 = None, None
for d in [oof_dir, data_dir, Path('/content')]:
    p = d / 'oof_mlp_v17a.npy'
    if p.exists() and oof_mlp17 is None:
        oof_mlp17 = np.load(p)
    p = d / 'test_mlp_v17a.npy'
    if p.exists() and test_mlp17 is None:
        test_mlp17 = np.load(p)
if oof_mlp17 is not None:
    print(f'v17a MLP: oof={oof_mlp17.shape}, test={test_mlp17.shape}')

# --- v19 Step3 MLP ---
oof_s3, test_s3 = None, None
for d in [oof_dir, data_dir, Path('/content')]:
    p = d / 'oof_mlp_s3_v19.npy'
    if p.exists() and oof_s3 is None:
        oof_s3 = np.load(p)
    p = d / 'test_mlp_s3_v19.npy'
    if p.exists() and test_s3 is None:
        test_s3 = np.load(p)
if oof_s3 is not None:
    print(f'Step3 MLP: oof={oof_s3.shape}, test={test_s3.shape}')

# --- v19b ベースライン構築 ---
# v19b best: w_v10b=0.78, w_v17a=0.08, w_s3=0.14
if oof_s3 is not None and oof_mlp17 is not None:
    oof_base = 0.78 * oof_v10b + 0.08 * oof_mlp17 + 0.14 * oof_s3
    test_base = 0.78 * test_v10b + 0.08 * test_mlp17 + 0.14 * test_s3
    print(f'\nv19b (3モデル): w_v10b=0.78, w_v17a=0.08, w_s3=0.14')
elif oof_mlp17 is not None:
    # v17a fallback
    best_w, best_cv = 0, 999
    for w in np.arange(0.02, 0.31, 0.01):
        b = (1-w) * oof_v10b + w * oof_mlp17
        cv = mean_squared_error(y_true, b)
        if cv < best_cv: best_cv = cv; best_w = w
    oof_base = (1-best_w) * oof_v10b + best_w * oof_mlp17
    test_base = (1-best_w) * test_v10b + best_w * test_mlp17
    print(f'\nv17a (2モデル): w_v10b={1-best_w:.2f}, w_mlp={best_w:.2f}')
else:
    oof_base = oof_v10b.copy()
    test_base = test_v10b.copy()
    print(f'\nv10b単体')

baseline_cv = mean_squared_error(y_true, oof_base)
print(f'ベースライン CV = {baseline_cv:.6f}')

In [ ]:
## Cell 4: 予測値の分布確認
print('=' * 60)
print('[予測値 vs 真値 の分布比較]')
print('=' * 60)

print(f'\n[真値 lever]')
print(f'  mean={y_true.mean():.4f}, std={y_true.std():.4f}')
print(f'  min={y_true.min():.2f}, max={y_true.max():.2f}')
for thr in [1, 2, 5, 10]:
    n = (np.abs(y_true) > thr).sum()
    print(f'  |y|>{thr}: {n:,} ({n/len(y_true)*100:.1f}%)')

print(f'\n[v19b予測]')
print(f'  mean={oof_base.mean():.4f}, std={oof_base.std():.4f}')
print(f'  min={oof_base.min():.2f}, max={oof_base.max():.2f}')
for thr in [1, 2, 5, 10]:
    n = (np.abs(oof_base) > thr).sum()
    print(f'  |ŷ|>{thr}: {n:,} ({n/len(oof_base)*100:.1f}%)')

# 振幅の比較
print(f'\n[振幅比 (予測/真値)]')
print(f'  std比: {oof_base.std() / y_true.std():.4f}')
# Active帯のみ
active = np.abs(y_true) > 0.1
print(f'  Active帯 std比: {oof_base[active].std() / y_true[active].std():.4f}')
print(f'  → 1.0未満なら予測が「ぬるい」（谷が浅い）')

---
## Step 1: Power Transformation

In [ ]:
## Cell 5: Power Transformation grid search
print('=' * 60)
print('[Step 1] Power Transformation: sign(y) × |y|^p')
print('=' * 60)

def power_transform(pred, p):
    """sign(y) × |y|^p: p>1で谷を強調、p<1で圧縮"""
    return np.sign(pred) * np.abs(pred) ** p

# 細かいgrid searchで最適pを探索
p_candidates = np.arange(0.80, 1.51, 0.005)
results_power = []

for p in p_candidates:
    oof_pt = power_transform(oof_base, p)
    cv = mean_squared_error(y_true, oof_pt)
    results_power.append((p, cv))

# ベスト
results_power.sort(key=lambda x: x[1])
best_p, best_cv_pt = results_power[0]
print(f'\nBest p = {best_p:.3f}')
print(f'CV: {baseline_cv:.6f} -> {best_cv_pt:.6f} (delta={best_cv_pt - baseline_cv:+.6f})')

# Top 10
print(f'\n[Top 10]')
for p, cv in results_power[:10]:
    print(f'  p={p:.3f}: CV={cv:.6f} (delta={cv - baseline_cv:+.6f})')

# p=1.0（変換なし）の位置
cv_at_1 = [cv for p, cv in results_power if abs(p - 1.0) < 0.001][0]
print(f'\n  p=1.000: CV={cv_at_1:.6f} (変換なし = ベースライン)')

In [ ]:
## Cell 6: Power Transform の帯域別効果分析
print('=' * 60)
print(f'[Step 1] Power Transform p={best_p:.3f} の帯域別効果')
print('=' * 60)

oof_pt_best = power_transform(oof_base, best_p)

# 帯域別MSE比較
bands = [
    ('Quiet |y|≤0.1', np.abs(y_true) <= 0.1),
    ('Small 0.1<|y|≤1', (np.abs(y_true) > 0.1) & (np.abs(y_true) <= 1)),
    ('Medium 1<|y|≤5', (np.abs(y_true) > 1) & (np.abs(y_true) <= 5)),
    ('Large |y|>5', np.abs(y_true) > 5),
]

print(f'{"帯域":<20s} {"N":>8s} {"MSE_before":>12s} {"MSE_after":>12s} {"delta":>10s} {"寄与%":>6s}')
print('-' * 70)
for name, mask in bands:
    n = mask.sum()
    if n == 0: continue
    mse_before = np.mean((oof_base[mask] - y_true[mask]) ** 2)
    mse_after = np.mean((oof_pt_best[mask] - y_true[mask]) ** 2)
    contribution = (mse_before * n) / (baseline_cv * len(y_true)) * 100
    delta = mse_after - mse_before
    print(f'{name:<20s} {n:>8,d} {mse_before:>12.4f} {mse_after:>12.4f} {delta:>+10.4f} {contribution:>5.1f}%')

---
## Step 2: Session Scaling

In [ ]:
## Cell 7: Session Scaling（session = sample_id）
print('=' * 60)
print('[Step 2] Session Scaling')
print('=' * 60)

# まずPower Transform適用後の予測をベースにする
if best_cv_pt < baseline_cv:
    oof_input = oof_pt_best
    print(f'入力: Power Transform済み (p={best_p:.3f})')
    cv_input = best_cv_pt
else:
    oof_input = oof_base.copy()
    print(f'入力: ベースライン（Power Transformなし）')
    cv_input = baseline_cv

# --- 方法A: sample_id単位スケーリング ---
print(f'\n[A] sample_id単位スケーリング')
print(f'  各sample_idの予測に最適なスケール係数αを求める')

train_sids = train_df['sample_id'].values
unique_sids_train = train_df['sample_id'].unique()

# sample_id単位で最適スケールを計算
# OOFで最適α: argmin_α Σ(α*ŷ - y)^2 = Σ(ŷ*y) / Σ(ŷ^2)
oof_scaled_a = oof_input.copy()
alpha_per_sid = {}

for sid in unique_sids_train:
    mask = train_sids == sid
    pred_sid = oof_input[mask]
    true_sid = y_true[mask]
    denom = np.sum(pred_sid ** 2)
    if denom > 1e-10:
        alpha = np.sum(pred_sid * true_sid) / denom
        # 極端なスケーリングを防止
        alpha = np.clip(alpha, 0.5, 2.0)
    else:
        alpha = 1.0
    alpha_per_sid[sid] = alpha
    oof_scaled_a[mask] = pred_sid * alpha

cv_scaled_a = mean_squared_error(y_true, oof_scaled_a)
alphas = np.array(list(alpha_per_sid.values()))
print(f'  α統計: mean={alphas.mean():.3f}, std={alphas.std():.3f}, '
      f'min={alphas.min():.3f}, max={alphas.max():.3f}')
print(f'  CV: {cv_input:.6f} -> {cv_scaled_a:.6f} (delta={cv_scaled_a - cv_input:+.6f})')
print(f'  ⚠ 注意: これはOOFの各sample_idで直接最適化したので過学習の可能性大')

In [ ]:
## Cell 8: Session Scaling（CV整合性のある方法）
print('=' * 60)
print('[Step 2b] mouse_id単位スケーリング（CV-safe）')
print('=' * 60)

# mouse_id単位: train全体で学習 → testに適用可能
# GroupKFoldのfoldをまたいでmouseが分散するため、CV整合性が保てる
train_mice = train_df['mouse_id'].values
unique_mice = sorted(train_df['mouse_id'].unique())

# --- mouse_id単位の最適スケール ---
oof_mouse_scaled = oof_input.copy()
alpha_per_mouse = {}

for mid in unique_mice:
    mask = train_mice == mid
    pred_m = oof_input[mask]
    true_m = y_true[mask]
    denom = np.sum(pred_m ** 2)
    if denom > 1e-10:
        alpha = np.sum(pred_m * true_m) / denom
        alpha = np.clip(alpha, 0.7, 1.5)
    else:
        alpha = 1.0
    alpha_per_mouse[mid] = alpha
    oof_mouse_scaled[mask] = pred_m * alpha

cv_mouse = mean_squared_error(y_true, oof_mouse_scaled)
m_alphas = np.array(list(alpha_per_mouse.values()))
print(f'  α統計: mean={m_alphas.mean():.3f}, std={m_alphas.std():.3f}')
print(f'  CV: {cv_input:.6f} -> {cv_mouse:.6f} (delta={cv_mouse - cv_input:+.6f})')
print(f'  ⚠ 同一mouseがtrain/valに跨る場合、過学習リスクあり')

# --- mouse_id × day_n 単位 ---
print(f'\n[Step 2c] mouse_id × day_n 単位スケーリング')
train_days = train_df['day_n'].values
oof_md_scaled = oof_input.copy()
alpha_per_md = {}

for mid in unique_mice:
    for day in sorted(train_df.loc[train_df['mouse_id']==mid, 'day_n'].unique()):
        mask = (train_mice == mid) & (train_days == day)
        if mask.sum() == 0: continue
        pred_md = oof_input[mask]
        true_md = y_true[mask]
        denom = np.sum(pred_md ** 2)
        if denom > 1e-10:
            alpha = np.sum(pred_md * true_md) / denom
            alpha = np.clip(alpha, 0.7, 1.5)
        else:
            alpha = 1.0
        alpha_per_md[(mid, day)] = alpha
        oof_md_scaled[mask] = pred_md * alpha

cv_md = mean_squared_error(y_true, oof_md_scaled)
md_alphas = np.array(list(alpha_per_md.values()))
print(f'  α統計: mean={md_alphas.mean():.3f}, std={md_alphas.std():.3f}')
print(f'  セッション数: {len(alpha_per_md)}')
print(f'  CV: {cv_input:.6f} -> {cv_md:.6f} (delta={cv_md - cv_input:+.6f})')
print(f'  ⚠ 過学習リスク高（セッション数が多い）')

In [ ]:
## Cell 9: CV-safe Session Scaling（LeaveOneGroupOut検証）
print('=' * 60)
print('[Step 2d] CV-safe Session Scaling（fold別検証）')
print('=' * 60)

from sklearn.model_selection import GroupKFold

# v17aと同一fold分割を再現
gkf = GroupKFold(n_splits=5)
unique_sids = train_df['sample_id'].unique()
sid_splits = list(gkf.split(np.arange(len(unique_sids)), groups=unique_sids))
sid_to_fold = {}
for fold, (_, vai) in enumerate(sid_splits):
    for idx in vai:
        sid_to_fold[unique_sids[idx]] = fold
row_folds = np.array([sid_to_fold[sid] for sid in train_sids])

# --- グローバルα（全train→1つのα）のCV検証 ---
print(f'\n[Global α] 全trainで1つのスケール係数')
best_cv_global, best_alpha_global = cv_input, 1.0
for alpha in np.arange(0.80, 1.30, 0.005):
    cv = mean_squared_error(y_true, oof_input * alpha)
    if cv < best_cv_global:
        best_cv_global = cv
        best_alpha_global = alpha
print(f'  Best α={best_alpha_global:.3f}')
print(f'  CV: {cv_input:.6f} -> {best_cv_global:.6f} (delta={best_cv_global - cv_input:+.6f})')

# --- mouse単位α のCV-safe検証 ---
# fold内のtrainデータでmouse別αを学習 → valデータに適用
print(f'\n[Mouse α CV-safe] fold内train→val適用')
oof_mouse_cvsafe = oof_input.copy()

for fold in range(5):
    tr_mask = row_folds != fold
    va_mask = row_folds == fold

    # train foldでmouse別αを学習
    fold_alpha = {}
    for mid in unique_mice:
        m_tr = tr_mask & (train_mice == mid)
        if m_tr.sum() == 0:
            fold_alpha[mid] = 1.0
            continue
        pred_tr = oof_input[m_tr]
        true_tr = y_true[m_tr]
        denom = np.sum(pred_tr ** 2)
        if denom > 1e-10:
            alpha = np.sum(pred_tr * true_tr) / denom
            alpha = np.clip(alpha, 0.7, 1.5)
        else:
            alpha = 1.0
        fold_alpha[mid] = alpha

    # val foldに適用
    for mid in unique_mice:
        m_va = va_mask & (train_mice == mid)
        if m_va.sum() == 0: continue
        oof_mouse_cvsafe[m_va] = oof_input[m_va] * fold_alpha[mid]

cv_mouse_cvsafe = mean_squared_error(y_true, oof_mouse_cvsafe)
print(f'  CV: {cv_input:.6f} -> {cv_mouse_cvsafe:.6f} (delta={cv_mouse_cvsafe - cv_input:+.6f})')

# --- mouse × day αのCV-safe検証 ---
print(f'\n[Mouse×Day α CV-safe] fold内train→val適用')
oof_md_cvsafe = oof_input.copy()

for fold in range(5):
    tr_mask = row_folds != fold
    va_mask = row_folds == fold

    fold_alpha_md = {}
    for mid in unique_mice:
        mouse_days = sorted(train_df.loc[train_df['mouse_id']==mid, 'day_n'].unique())
        for day in mouse_days:
            m_tr = tr_mask & (train_mice == mid) & (train_days == day)
            if m_tr.sum() == 0:
                fold_alpha_md[(mid, day)] = 1.0
                continue
            pred_tr = oof_input[m_tr]
            true_tr = y_true[m_tr]
            denom = np.sum(pred_tr ** 2)
            if denom > 1e-10:
                alpha = np.sum(pred_tr * true_tr) / denom
                alpha = np.clip(alpha, 0.7, 1.5)
            else:
                alpha = 1.0
            fold_alpha_md[(mid, day)] = alpha

    for mid in unique_mice:
        mouse_days = sorted(train_df.loc[train_df['mouse_id']==mid, 'day_n'].unique())
        for day in mouse_days:
            m_va = va_mask & (train_mice == mid) & (train_days == day)
            if m_va.sum() == 0: continue
            a = fold_alpha_md.get((mid, day), 1.0)
            oof_md_cvsafe[m_va] = oof_input[m_va] * a

cv_md_cvsafe = mean_squared_error(y_true, oof_md_cvsafe)
print(f'  CV: {cv_input:.6f} -> {cv_md_cvsafe:.6f} (delta={cv_md_cvsafe - cv_input:+.6f})')

---
## Step 3: 最終組み合わせ

In [ ]:
## Cell 10: 全組み合わせの比較 + 最終選択
print('=' * 60)
print('[Step 3] 全後処理の比較')
print('=' * 60)

results = [
    ('ベースライン (v19b)',       baseline_cv,       'none'),
    (f'Power p={best_p:.3f}',     best_cv_pt,        'power'),
    (f'Global α={best_alpha_global:.3f}', best_cv_global, 'global_alpha'),
    ('Mouse α (CV-safe)',         cv_mouse_cvsafe,   'mouse_alpha'),
    ('Mouse×Day α (CV-safe)',     cv_md_cvsafe,      'md_alpha'),
]

# Power + Global α の組み合わせ
best_cv_pg, best_pg_alpha = best_cv_pt, 1.0
for alpha in np.arange(0.80, 1.30, 0.005):
    cv = mean_squared_error(y_true, oof_pt_best * alpha)
    if cv < best_cv_pg:
        best_cv_pg = cv; best_pg_alpha = alpha
results.append((f'Power + Global α={best_pg_alpha:.3f}', best_cv_pg, 'power_global'))

# Power + Mouse α CV-safe
oof_power_mouse = oof_pt_best.copy()
for fold in range(5):
    tr_mask = row_folds != fold
    va_mask = row_folds == fold
    for mid in unique_mice:
        m_tr = tr_mask & (train_mice == mid)
        if m_tr.sum() == 0: continue
        pred_tr = oof_pt_best[m_tr]
        true_tr = y_true[m_tr]
        denom = np.sum(pred_tr ** 2)
        alpha = np.clip(np.sum(pred_tr * true_tr) / max(denom, 1e-10), 0.7, 1.5)
        m_va = va_mask & (train_mice == mid)
        if m_va.sum() == 0: continue
        oof_power_mouse[m_va] = oof_pt_best[m_va] * alpha
cv_power_mouse = mean_squared_error(y_true, oof_power_mouse)
results.append(('Power + Mouse α (CV-safe)', cv_power_mouse, 'power_mouse'))

# Power + Mouse×Day α CV-safe
oof_power_md = oof_pt_best.copy()
for fold in range(5):
    tr_mask = row_folds != fold
    va_mask = row_folds == fold
    for mid in unique_mice:
        mouse_days = sorted(train_df.loc[train_df['mouse_id']==mid, 'day_n'].unique())
        for day in mouse_days:
            m_tr = tr_mask & (train_mice == mid) & (train_days == day)
            if m_tr.sum() == 0: continue
            pred_tr = oof_pt_best[m_tr]
            true_tr = y_true[m_tr]
            denom = np.sum(pred_tr ** 2)
            alpha = np.clip(np.sum(pred_tr * true_tr) / max(denom, 1e-10), 0.7, 1.5)
            m_va = va_mask & (train_mice == mid) & (train_days == day)
            if m_va.sum() == 0: continue
            oof_power_md[m_va] = oof_pt_best[m_va] * alpha
cv_power_md = mean_squared_error(y_true, oof_power_md)
results.append(('Power + Mouse×Day α (CV-safe)', cv_power_md, 'power_md'))

# ランキング表示
results.sort(key=lambda x: x[1])
print(f'\n{"Rank":<5s} {"手法":<35s} {"CV":>12s} {"delta":>12s}')
print('-' * 66)
for i, (name, cv, tag) in enumerate(results):
    delta = cv - baseline_cv
    marker = ' ★' if i == 0 and delta < 0 else ''
    print(f'{i+1:<5d} {name:<35s} {cv:>12.6f} {delta:>+12.6f}{marker}')

# ベスト設定を記録
best_name, best_cv_final, best_tag = results[0]
print(f'\n=== 最良: {best_name} CV={best_cv_final:.6f} ===')

In [ ]:
## Cell 11: テスト予測生成 + Submission
print('=' * 60)
print('[Submission] 最良設定でテスト予測生成')
print('=' * 60)

# --- Power Transform をテストに適用 ---
if best_tag in ['power', 'power_global', 'power_mouse', 'power_md']:
    test_pt = power_transform(test_base, best_p)
    print(f'  Power Transform: p={best_p:.3f}')
else:
    test_pt = test_base.copy()
    print(f'  Power Transform: なし')

# --- Scaling をテストに適用 ---
test_sids = test_df['sample_id'].values
test_mice = test_df['mouse_id'].values
test_days = test_df['day_n'].values

if best_tag == 'none':
    test_final = test_base.copy()
    print(f'  Scaling: なし')

elif best_tag == 'power':
    test_final = test_pt.copy()
    print(f'  Scaling: なし (Power only)')

elif best_tag == 'global_alpha':
    test_final = oof_input.copy()  # oof_inputのテスト版
    # Power適用済みかチェック
    if best_cv_pt < baseline_cv:
        test_final = power_transform(test_base, best_p) * best_alpha_global
    else:
        test_final = test_base * best_alpha_global
    print(f'  Scaling: Global α={best_alpha_global:.3f}')

elif best_tag == 'power_global':
    test_final = test_pt * best_pg_alpha
    print(f'  Scaling: Global α={best_pg_alpha:.3f}')

elif best_tag in ['mouse_alpha', 'power_mouse']:
    # 全trainでmouse別αを再計算
    base_for_alpha = oof_pt_best if 'power' in best_tag else oof_input
    test_input = test_pt if 'power' in best_tag else test_base
    final_alpha_mouse = {}
    for mid in unique_mice:
        mask = train_mice == mid
        pred_m = base_for_alpha[mask]
        true_m = y_true[mask]
        denom = np.sum(pred_m ** 2)
        alpha = np.clip(np.sum(pred_m * true_m) / max(denom, 1e-10), 0.7, 1.5)
        final_alpha_mouse[mid] = alpha

    test_final = test_input.copy()
    for mid in sorted(test_df['mouse_id'].unique()):
        mask = test_mice == mid
        a = final_alpha_mouse.get(mid, 1.0)
        test_final[mask] = test_input[mask] * a
    print(f'  Scaling: Mouse α')

elif best_tag in ['md_alpha', 'power_md']:
    # 全trainでmouse×day別αを再計算
    base_for_alpha = oof_pt_best if 'power' in best_tag else oof_input
    test_input = test_pt if 'power' in best_tag else test_base
    final_alpha_md = {}
    for mid in unique_mice:
        mouse_days = sorted(train_df.loc[train_df['mouse_id']==mid, 'day_n'].unique())
        for day in mouse_days:
            mask = (train_mice == mid) & (train_days == day)
            if mask.sum() == 0: continue
            pred_md = base_for_alpha[mask]
            true_md = y_true[mask]
            denom = np.sum(pred_md ** 2)
            alpha = np.clip(np.sum(pred_md * true_md) / max(denom, 1e-10), 0.7, 1.5)
            final_alpha_md[(mid, day)] = alpha

    # mouse平均α（testにtrain外のdayがある場合のfallback）
    mouse_mean_alpha = {}
    for mid in unique_mice:
        mid_alphas = [v for (m, d), v in final_alpha_md.items() if m == mid]
        mouse_mean_alpha[mid] = np.mean(mid_alphas) if mid_alphas else 1.0

    test_final = test_input.copy()
    for mid in sorted(test_df['mouse_id'].unique()):
        test_mouse_days = sorted(test_df.loc[test_df['mouse_id']==mid, 'day_n'].unique())
        for day in test_mouse_days:
            mask = (test_mice == mid) & (test_days == day)
            if mask.sum() == 0: continue
            a = final_alpha_md.get((mid, day), mouse_mean_alpha.get(mid, 1.0))
            test_final[mask] = test_input[mask] * a
    print(f'  Scaling: Mouse×Day α')

# --- Submission ---
sub = test_df[['id']].copy()
sub['lever'] = test_final
sub_name = 'submission_v21.csv'
sub.to_csv(sub_name, index=False)
print(f'\n[OUT] {sub_name}')
print(f'  mean={sub["lever"].mean():.4f}, std={sub["lever"].std():.4f}, shape={sub.shape}')
print(f'  手法: {best_name}')
print(f'  CV: {baseline_cv:.6f} -> {best_cv_final:.6f} (delta={best_cv_final - baseline_cv:+.6f})')

from google.colab import files
files.download(sub_name)

In [ ]:
## Cell 12: Power未適用版のsubmission（比較用）
# Power Transformの効果が不明確な場合、ベースラインsubmissionも作る
if best_tag != 'none':
    sub_base = test_df[['id']].copy()
    sub_base['lever'] = test_base
    sub_base.to_csv('submission_v19b.csv', index=False)
    print(f'比較用: submission_v19b.csv (ベースライン CV={baseline_cv:.6f})')
    from google.colab import files
    files.download('submission_v19b.csv')

In [ ]:
## Cell 13: サマリー
print(f'\n{"="*60}')
print(f'[v21 サマリー]')
print(f'  ベースライン (v19b): CV={baseline_cv:.6f}, LB=0.707')
print(f'  Power Transform:     p={best_p:.3f}, CV={best_cv_pt:.6f} (delta={best_cv_pt - baseline_cv:+.6f})')
print(f'  Global α:            α={best_alpha_global:.3f}, CV={best_cv_global:.6f}')
print(f'  Mouse α (CV-safe):   CV={cv_mouse_cvsafe:.6f}')
print(f'  Mouse×Day (CV-safe): CV={cv_md_cvsafe:.6f}')
print(f'  ---')
print(f'  最良: {best_name}')
print(f'  最良CV: {best_cv_final:.6f} (delta={best_cv_final - baseline_cv:+.6f})')
print(f'{"="*60}')

## 実験メモ

### v21: Post-Processing Pipeline
新規モデル学習なし。v19bの予測値を変形するだけ。

### Power Transformation
- `sign(y) × |y|^p`: p>1で谷を強調
- アンサンブルの「角丸め」を補正
- MSEは二乗誤差なので、大きな予測の改善が効く

### Session Scaling
- Global α: 全体のスケール補正
- Mouse α: 個体差の補正
- Mouse×Day α: セッション単位の補正
- CV-safe: fold内trainでα学習→valに適用

### 過学習リスク
- sample_id単位: 最も過学習しやすい（使用禁止）
- mouse×day: セッション数多く過学習リスク中
- mouse: 個体数少なく比較的安全
- global: 最も安全（パラメータ1つだけ）